# Module 2: Modular Code & Resource Modeling

**Concepts:** Functions · arguments (`*args`, `**kwargs`) · lambdas · classes · methods · inheritance

**Audience:** DevOps / Platform / MLOps — turn one-off scripts into reusable pieces and model clusters, jobs, or nodes as objects.

---

### How to use this notebook

| Markdown | Read theory and where it applies in infra tooling. |
| Code | Edit, run (`Shift+Enter`), break things on purpose. |

**Pattern:** same as Module 1 — short explanation → tiny example → next topic.


## 1. Why functions?

Copy-pasting the same `if env == "prod"` block across a 400-line deploy script is how incidents happen.

A **function** names a reusable action: inputs → output (or side effect). Your CI image, internal CLI, and notebook cells all share the same idea as bash functions — but with real parameters and return values.


In [ ]:
def connection_banner(host: str, port: int, protocol: str = "ssh") -> str:
    return f"{protocol.upper()} {host}:{port}"


print(connection_banner("10.8.116.50", 22))
print(connection_banner("10.8.116.50", 6443, protocol="https"))


**Platform:** one function formats log lines for every playbook step.
**MLOps:** one function validates metric dicts before register-model.


## 2. Parameters: defaults, keyword args, return

- **Default values** — optional behaviour (`dry_run=True`).
- **Keyword arguments** — call with names for clarity in long signatures.
- **`return`** — send a result back; no return means `None`.


In [ ]:
def rollout_action(env: str, replicas: int, dry_run: bool = False) -> str:
    action = f"scale to {replicas} in {env}"
    if dry_run:
        return f"[DRY-RUN] would {action}"
    return action


print(rollout_action("staging", 3))
print(rollout_action(env="prod", replicas=5, dry_run=True))


## 3. `*args` — variable number of positional arguments

Use when you do not know how many hosts, paths, or stages you will pass.

`*hosts` collects extra positional args into a **tuple** inside the function.


In [ ]:
def ping_plan(*hosts: str) -> list[str]:
    return [f"check {h}" for h in hosts]


print(ping_plan("node-a"))
print(ping_plan("node-a", "node-b", "node-c"))


## 4. `**kwargs` — variable keyword options

Common for wrapping APIs or Helm-style overrides: unknown key/value pairs land in a **dict**.


In [ ]:
def job_summary(name: str, **labels) -> dict:
    return {"job": name, "labels": labels}


print(job_summary("train-fraud", env="staging", team="ml-platform", gpu="true"))


## 5. Combining `*args` and `**kwargs`

Order in definition: normal params → `*args` → keyword-only → `**kwargs`.

**Footgun:** shadowing — do not name a kwarg `labels` if you also unpack `**labels`.


In [ ]:
def emit_events(event_type: str, *targets, **meta):
    for t in targets:
        print(event_type, t, meta)


emit_events("restart", "api-0", "api-1", reason="memory", ticket="OPS-42")


## 6. Lambda — tiny one-expression functions

`lambda x: x * 2` is an anonymous function. Fine for **sort keys** and short callbacks; for anything readable, use `def`.

**Platform example:** sort nodes by load.


In [ ]:
nodes = [{"name": "a", "load": 1.2}, {"name": "b", "load": 0.4}]
for n in sorted(nodes, key=lambda x: x["load"]):
    print(n["name"], n["load"])


## 7. Classes — model a resource

A **class** is a blueprint; an **instance** is one real object (one cluster, one pipeline job).

**Attributes** = data. **Methods** = functions on that data (`self` is the instance).


In [ ]:
class Node:
    def __init__(self, name: str, ip: str, role: str):
        self.name = name
        self.ip = ip
        self.role = role
        self.maintenance = False

    def cordon_summary(self) -> str:
        state = "MAINT" if self.maintenance else "ACTIVE"
        return f"{self.name} {self.ip} [{self.role}] {state}"


n = Node("worker-2", "10.0.0.12", "worker")
n.maintenance = True
print(n.cordon_summary())


## 8. Modelling CI/CD or training jobs

Same pattern: constructor sets identity; methods encode behaviour.


In [ ]:
class PipelineJob:
    def __init__(self, project: str, ref: str):
        self.project = project
        self.ref = ref
        self.status = "pending"

    def mark(self, status: str) -> None:
        self.status = status

    def __repr__(self) -> str:
        return f"PipelineJob({self.project}@{self.ref}={self.status})"


job = PipelineJob("ml/train", "main")
job.mark("running")
print(job)


## 9. Inheritance — specialise without copy-paste

Subclass **extends** a base class: shared behaviour in parent, overrides in child.

**Platform:** `ProdCluster` vs `LabCluster` with different defaults.


In [ ]:
class Cluster:
    def __init__(self, name: str, region: str):
        self.name = name
        self.region = region

    def policy(self) -> str:
        return "standard change window"


class ProdCluster(Cluster):
    def policy(self) -> str:
        return "change board + maintenance window required"


for c in [Cluster("ncs-dev", "eu"), ProdCluster("ncs57", "eu")]:
    print(c.name, "->", c.policy())


## 10. Capstone — functions + class together

Parse input with functions; store state in objects; loop and decide.


In [ ]:
def parse_host_line(line: str) -> tuple[str, str]:
    name, ip = line.split("=", maxsplit=1)
    return name.strip(), ip.strip()


class Inventory:
    def __init__(self):
        self.nodes: list[Node] = []

    def add_line(self, line: str, role: str = "worker") -> None:
        name, ip = parse_host_line(line)
        self.nodes.append(Node(name, ip, role))

    def all_summaries(self) -> list[str]:
        return [n.cordon_summary() for n in self.nodes]


inv = Inventory()
for raw in ["web-1=10.0.0.5", "web-2=10.0.0.6"]:
    inv.add_line(raw, role="app")
for s in inv.all_summaries():
    print(s)


---
# Exercises

Markdown = task. Code cell = `# TODO`. Solutions at bottom.


**Ex 1** — Function `in_cidr(ip_octets: tuple[int,...]) -> bool` returning True if first octet is 10 (simple lab rule). Test with `(10,0,0,1)` and `(192,168,1,1)`.


In [ ]:
# TODO


**Ex 2** — Function `merge_labels(base: dict, **extra) -> dict` returning new dict (copy base, update with extra).


In [ ]:
# TODO


**Ex 3** — Class `Service` with `name`, `port`, method `url(host)` returning `http://{host}:{port}`.


In [ ]:
# TODO


**Ex 4** — Subclass `CriticalService(Service)` overriding `url` to prefix `[CRIT]` in the returned string (call super or rebuild).


In [ ]:
# TODO


**Ex 5** — Use `sorted` + `lambda` to order dicts `pods` by key `restarts` descending.


In [ ]:
pods = [
    {"name": "a", "restarts": 1},
    {"name": "b", "restarts": 5},
    {"name": "c", "restarts": 2},
]
# TODO


---
# Solutions


In [ ]:
def in_cidr(octets: tuple[int, ...]) -> bool:
    return len(octets) >= 1 and octets[0] == 10


print(in_cidr((10, 0, 0, 1)), in_cidr((192, 168, 1, 1)))


In [ ]:
def merge_labels(base: dict, **extra) -> dict:
    out = dict(base)
    out.update(extra)
    return out


print(merge_labels({"app": "api"}, env="prod", team="platform"))


In [ ]:
class Service:
    def __init__(self, name: str, port: int):
        self.name = name
        self.port = port

    def url(self, host: str) -> str:
        return f"http://{host}:{self.port}"


print(Service("api", 8080).url("10.0.0.5"))


In [ ]:
class CriticalService(Service):
    def url(self, host: str) -> str:
        return f"[CRIT] {super().url(host)}"


print(CriticalService("db", 5432).url("10.0.0.7"))


In [ ]:
pods = [
    {"name": "a", "restarts": 1},
    {"name": "b", "restarts": 5},
    {"name": "c", "restarts": 2},
]
for p in sorted(pods, key=lambda x: x["restarts"], reverse=True):
    print(p["name"], p["restarts"])


**Next:** Module 3 — generators for huge logs and decorators for retries.
